# تحليل الطلب والعروض والتقلب الأسبوعي

دفتر قابل لإعادة التشغيل لتحليل بيانات التدريب في `data/row/train.csv` والإجابة عن ثلاثة أسئلة: ارتباط الترويج بالطلب، وعلاقة فرق السعر بالطلب، والوجبات الأعلى تقلبًا أسبوعيًا.

**حدود التفسير:** البيانات رصدية، لذلك لا تثبت السببية. لا يوجد في المصدر عمود خصم موثق؛ الفرق بين `base_price` و`checkout_price` يستخدم هنا كمؤشر سعري استكشافي فقط. كذلك لا يعني غياب سجل وجبة في أسبوع أن الطلب كان صفرًا، لذا يحسب التقلب على الأسابيع المسجلة فقط. مؤشرات المخزون الواردة وصفية وليست توصية شراء أو مستوى مخزون أمان.

In [1]:
from pathlib import Path
import math

import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next((path for path in candidates if (path / "data" / "row" / "train.csv").exists()), None)
if project_root is None:
    raise FileNotFoundError("لم أجد data/row/train.csv انطلاقًا من مجلد العمل الحالي")

data_dir = project_root / "data" / "row"
train = pd.read_csv(data_dir / "train.csv")
meals = pd.read_csv(data_dir / "meal_info.csv")
centers = pd.read_csv(data_dir / "fulfilment_center_info.csv")

required = {
    "week", "center_id", "meal_id", "checkout_price", "base_price",
    "emailer_for_promotion", "homepage_featured", "num_orders",
}
missing_columns = required - set(train.columns)
if missing_columns:
    raise ValueError(f"أعمدة مطلوبة غير موجودة: {sorted(missing_columns)}")
if train[list(required)].isna().any().any():
    raise ValueError("توجد قيم مفقودة في أعمدة التحليل؛ افحصها قبل المتابعة")
for column in ["emailer_for_promotion", "homepage_featured"]:
    if not set(train[column].unique()).issubset({0, 1}):
        raise ValueError(f"القيم في {column} ليست ثنائية")
if train.duplicated(["week", "center_id", "meal_id"]).any():
    raise ValueError("توجد مفاتيح (week, center_id, meal_id) مكررة")

print(f"صفوف التدريب: {len(train):,}؛ الأسابيع: {train.week.min()}–{train.week.max()}")
print(f"القيم المفقودة في أعمدة التحليل: {int(train[list(required)].isna().sum().sum()):,}")
print("تكرارات المفتاح الأسبوع/المركز/الوجبة:", int(train.duplicated(["week", "center_id", "meal_id"]).sum()))

unmatched_meals = int((~train.meal_id.isin(meals.meal_id)).sum())
before_rows = len(train)
orders = train.merge(meals, on="meal_id", how="left", validate="many_to_one")
print(f"ربط الوجبات: قبل={before_rows:,}، بعد={len(orders):,}، صفوف غير مطابقة={unmatched_meals:,}")
unmatched_centers = int((~orders.center_id.isin(centers.center_id)).sum())
before_center_rows = len(orders)
orders = orders.merge(centers, on="center_id", how="left", validate="many_to_one")
print(f"ربط المراكز: قبل={before_center_rows:,}، بعد={len(orders):,}، صفوف غير مطابقة={unmatched_centers:,}")
if len(orders) != before_rows or unmatched_meals or unmatched_centers:
    raise ValueError("فشل تحقق ثبات الصفوف أو مطابقة المفاتيح في الربط المرجعي")

صفوف التدريب: 456,548؛ الأسابيع: 1–145
القيم المفقودة في أعمدة التحليل: 0
تكرارات المفتاح الأسبوع/المركز/الوجبة: 0
ربط الوجبات: قبل=456,548، بعد=456,548، صفوف غير مطابقة=0
ربط المراكز: قبل=456,548، بعد=456,548، صفوف غير مطابقة=0


## إجمالي الطلبات ومتوسط الطلب الأسبوعي

يُحسب إجمالي الطلب بجمع `num_orders` لجميع سجلات التدريب، ثم يُحسب متوسط إجمالي الطلب الأسبوعي بقسمة ذلك المجموع على عدد الأسابيع الفريدة الموجودة في التدريب. بذلك يكون المقام عدد الأسابيع وليس عدد السجلات.

In [3]:
weekly_orders = orders.groupby("week")["num_orders"].sum().sort_index()
total_orders = int(weekly_orders.sum())
number_of_weeks = int(weekly_orders.index.nunique())
average_weekly_orders = total_orders / number_of_weeks

print(f"إجمالي الطلبات المسجلة في التدريب: {total_orders:,}")
print(f"عدد الأسابيع الفريدة: {number_of_weeks}")
print(f"متوسط إجمالي الطلب الأسبوعي: {average_weekly_orders:,.2f}")

إجمالي الطلبات المسجلة في التدريب: 119,557,485
عدد الأسابيع الفريدة: 145
متوسط إجمالي الطلب الأسبوعي: 824,534.38


## تقدير إجمالي الإيرادات

نقدّر قيمة مبيعات الوجبات بجمع `checkout_price × num_orders` لكل سجل. الأسعار في الملف مسجلة بمنزلتين عشريتين، لذا يحوّلها الكود إلى وحدات من مئة قبل الضرب والجمع لتجنب أخطاء الكسور العشرية. الناتج بوحدة السعر الموجودة في البيانات؛ العملة غير موثقة، كما أن الحساب لا يشمل تكاليف أو رسومًا ولا يمثل صافي الربح.

In [4]:
price_in_cents = np.rint(orders["checkout_price"].to_numpy(dtype=float) * 100).astype(np.int64)
if not np.allclose(orders["checkout_price"].to_numpy(dtype=float) * 100, price_in_cents, atol=1e-7):
    raise ValueError("توجد أسعار بأكثر من منزلتين عشريتين؛ راجع دقة المصدر قبل احتساب الإيراد")

orders["revenue_cents"] = price_in_cents * orders["num_orders"].to_numpy(dtype=np.int64)
weekly_revenue_cents = orders.groupby("week")["revenue_cents"].sum().sort_index()
total_revenue = int(weekly_revenue_cents.sum()) / 100
number_of_revenue_weeks = int(weekly_revenue_cents.index.nunique())
average_weekly_revenue = total_revenue / number_of_revenue_weeks

print(f"إجمالي الإيرادات المقدّرة في التدريب: {total_revenue:,.2f} (بوحدة السعر في المصدر)")
print(f"عدد الأسابيع: {number_of_revenue_weeks}")
print(f"متوسط الإيرادات الأسبوعية المقدّرة: {average_weekly_revenue:,.2f} (بوحدة السعر في المصدر)")

إجمالي الإيرادات المقدّرة في التدريب: 31,922,771,040.11 (بوحدة السعر في المصدر)
عدد الأسابيع: 145
متوسط الإيرادات الأسبوعية المقدّرة: 220,157,041.66 (بوحدة السعر في المصدر)


## 1. ارتباط البريد والظهور بالطلب

يقدّر النموذج فرقًا خطيًا في عدد الطلبات لكل سجل، مع إدخال المؤشرين والتفاعل بينهما. تزال تأثيرات ثابتة لزوج المركز-الوجبة، والمركز-الأسبوع، والوجبة-الأسبوع، وتجمع الأخطاء المعيارية على مستوى المركز. فواصل الثقة واختبارات الفرضية تستند إلى عدد المراكز، لا إلى افتراض استقلال كل سجل. هذه تقديرات ارتباط معدّلة، وليست أثرًا سببيًا.

In [8]:
def factorize_pair(frame, columns):
    return pd.MultiIndex.from_frame(frame[columns]).factorize(sort=True)[0]

fixed_effect_groups = [
    factorize_pair(orders, ["center_id", "meal_id"]),
    factorize_pair(orders, ["center_id", "week"]),
    factorize_pair(orders, ["meal_id", "week"]),
]
fixed_effect_levels = [int(group.max()) + 1 for group in fixed_effect_groups]
center_clusters = pd.factorize(orders.center_id, sort=True)[0]
cluster_count = int(center_clusters.max()) + 1
email = orders.emailer_for_promotion.to_numpy(dtype=float)
homepage = orders.homepage_featured.to_numpy(dtype=float)
promotion_x = np.column_stack([email, homepage, email * homepage])
outcome = orders.num_orders.to_numpy(dtype=float)

def absorb_fixed_effects(values, tolerance=1e-8, max_iterations=1000):
    residuals = values.astype(float, copy=True)
    for _ in range(max_iterations):
        for group, level_count in zip(fixed_effect_groups, fixed_effect_levels):
            counts = np.bincount(group, minlength=level_count).astype(float)
            for column in range(residuals.shape[1]):
                means = np.bincount(group, weights=residuals[:, column], minlength=level_count) / counts
                residuals[:, column] -= means[group]
        largest_mean = max(
            np.max(np.abs(np.bincount(group, weights=residuals[:, column], minlength=level_count)
                          / np.bincount(group, minlength=level_count)))
            for group, level_count in zip(fixed_effect_groups, fixed_effect_levels)
            for column in range(residuals.shape[1])
        )
        if largest_mean < tolerance:
            return residuals
    raise RuntimeError("لم تتقارب إزالة التأثيرات الثابتة")

def beta_continued_fraction(a, b, x):
    qab, qap, qam = a + b, a + 1, a - 1
    c = 1.0
    denominator = 1 - qab * x / qap
    denominator = max(abs(denominator), 1e-300) * (1 if denominator >= 0 else -1)
    denominator = 1 / denominator
    result = denominator
    for step in range(1, 301):
        twice = 2 * step
        term = step * (b - step) * x / ((qam + twice) * (a + twice))
        denominator = 1 + term * denominator
        denominator = max(abs(denominator), 1e-300) * (1 if denominator >= 0 else -1)
        c = 1 + term / c
        c = max(abs(c), 1e-300) * (1 if c >= 0 else -1)
        denominator = 1 / denominator
        result *= denominator * c
        term = -(a + step) * (qab + step) * x / ((a + twice) * (qap + twice))
        denominator = 1 + term * denominator
        denominator = max(abs(denominator), 1e-300) * (1 if denominator >= 0 else -1)
        c = 1 + term / c
        c = max(abs(c), 1e-300) * (1 if c >= 0 else -1)
        denominator = 1 / denominator
        delta = denominator * c
        result *= delta
        if abs(delta - 1) < 3e-14:
            break
    return result

def regularized_beta(a, b, x):
    if x <= 0:
        return 0.0
    if x >= 1:
        return 1.0
    factor = math.exp(math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b)
                      + a * math.log(x) + b * math.log1p(-x))
    if x < (a + 1) / (a + b + 2):
        return factor * beta_continued_fraction(a, b, x) / a
    return 1 - factor * beta_continued_fraction(a, b, 1 - x) / b

def t_two_sided_p_value(t_value, degrees_freedom):
    x = degrees_freedom / (degrees_freedom + t_value**2)
    return regularized_beta(degrees_freedom / 2, 0.5, x)

residualized = absorb_fixed_effects(np.column_stack([outcome, promotion_x]))
y_residual, x_residual = residualized[:, 0], residualized[:, 1:]
gram = x_residual.T @ x_residual
coefficients = np.linalg.solve(gram, x_residual.T @ y_residual)
model_residual = y_residual - x_residual @ coefficients
bread = np.linalg.inv(gram)
meat = np.zeros_like(gram)
for cluster in range(cluster_count):
    mask = center_clusters == cluster
    score = x_residual[mask].T @ model_residual[mask]
    meat += np.outer(score, score)
parameter_count = sum(fixed_effect_levels) - (len(fixed_effect_levels) - 1) + len(coefficients)
small_sample_correction = (cluster_count / (cluster_count - 1)) * ((len(orders) - 1) / (len(orders) - parameter_count))
covariance = bread @ meat @ bread * small_sample_correction
standard_errors = np.sqrt(np.diag(covariance))
critical_value = 1.992  # t critical value, 76 degrees of freedom

contrasts = {
    "البريد فقط": np.array([1.0, 0.0, 0.0]),
    "الصفحة الرئيسية فقط": np.array([0.0, 1.0, 0.0]),
    "البريد والصفحة معًا": np.array([1.0, 1.0, 1.0]),
}
results = []
for label, contrast in contrasts.items():
    estimate = float(contrast @ coefficients)
    se = float(np.sqrt(contrast @ covariance @ contrast))
    t_value = estimate / se
    baseline = float(outcome.mean() - estimate)
    results.append({
        "المقارنة": label,
        "فرق الطلب/سجل": estimate,
        "فاصل 95% الأدنى": estimate - critical_value * se,
        "فاصل 95% الأعلى": estimate + critical_value * se,
        "t": t_value,
        "p": t_two_sided_p_value(abs(t_value), cluster_count - 1),
    })
promotion_results = pd.DataFrame(results)
display(promotion_results.drop(columns="p").round(4).assign(
    p=promotion_results["p"].map(lambda value: f"{value:.3e}" if value >= 0.001 else "< 0.001")
))
print(f"عدد المراكز المستخدمة لتجميع الأخطاء المعيارية: {cluster_count}")

,المقارنة,فرق الطلب/سجل,فاصل 95% الأدنى,فاصل 95% الأعلى,t,p
0,البريد فقط,123.7703,103.7714,143.7692,12.3282,< 0.001
1,الصفحة الرئيسية فقط,97.2775,85.9937,108.5612,17.1730,< 0.001
2,البريد والصفحة معًا,244.0733,221.3036,266.8430,21.3527,< 0.001


عدد المراكز المستخدمة لتجميع الأخطاء المعيارية: 77


## 2. علاقة فرق السعر بالطلب

يعرّف `فرق السعر النسبي` استكشافيًا بأنه `(base_price - checkout_price) / base_price`. القيمة الموجبة تعني أن سعر الدفع أقل من الأساسي، لكنها **لا تثبت** أن الفرق خصم؛ كما أن بعض سجلات المصدر يكون فيها سعر الدفع أعلى من الأساسي. الجدول التالي وصفي ويعرض الطلب بحسب اتجاه الفرق فقط.

In [5]:
if (orders.base_price <= 0).any():
    raise ValueError("تعذر حساب فرق السعر النسبي لوجود base_price غير موجب")
orders["relative_price_gap"] = (orders.base_price - orders.checkout_price) / orders.base_price
orders["price_relation"] = np.select(
    [orders.checkout_price < orders.base_price, orders.checkout_price == orders.base_price],
    ["سعر الدفع أقل من الأساسي", "السعران متساويان"],
    default="سعر الدفع أعلى من الأساسي",
)
price_summary = (orders.groupby("price_relation", observed=True)
                 .agg(عدد_السجلات=("num_orders", "size"), متوسط_الطلبات=("num_orders", "mean"),
                      وسيط_الطلبات=("num_orders", "median"),
                      متوسط_فرق_السعر_النسبي=("relative_price_gap", "mean"))
                 .reset_index())
display(price_summary.round(3))
print("ارتباط بيرسون الوصفي بين فرق السعر النسبي والطلبات:",
      round(float(orders[["relative_price_gap", "num_orders"]].corr().iloc[0, 1]), 4))
print("هذا ارتباط غير مضبوط ولا يثبت أثر الخصم أو السببية.")

,price_relation,عدد_السجلات,متوسط_الطلبات,وسيط_الطلبات,متوسط_فرق_السعر_النسبي
0,السعران متساويان,110760,207.538,121.0,0.000
1,سعر الدفع أعلى من الأساسي,116101,226.705,123.0,-0.007
2,سعر الدفع أقل من الأساسي,229687,305.851,161.0,0.117


ارتباط بيرسون الوصفي بين فرق السعر النسبي والطلبات: 0.2052
هذا ارتباط غير مضبوط ولا يثبت أثر الخصم أو السببية.


## 3. الوجبات الأعلى تقلبًا أسبوعيًا

نجمع الطلبات لكل وجبة في كل أسبوع مرصود، ثم نحسب المتوسط والانحراف المعياري ومعامل الاختلاف (الانحراف المعياري ÷ المتوسط) وأعلى أسبوع نسبةً إلى المتوسط. يعرض الترتيب أعلى معاملات اختلاف مع عدد الأسابيع المرصودة؛ لا تعوّض الأسابيع الغائبة بصفر لأن غياب السجل لا يثبت انعدام الطلب. معامل الاختلاف المرتفع يحدد تذبذبًا نسبيًا، أما الاحتياطي الفعلي فيحتاج زمن التوريد ومستوى الخدمة ومعلومات نفاد المخزون.

In [6]:
weekly_meal_demand = (orders.groupby(["meal_id", "week"], as_index=False)
                      .agg(weekly_orders=("num_orders", "sum")))
volatility = (weekly_meal_demand.groupby("meal_id")
              .agg(weeks_observed=("week", "nunique"), mean_weekly_orders=("weekly_orders", "mean"),
                   std_weekly_orders=("weekly_orders", "std"), max_weekly_orders=("weekly_orders", "max"))
              .reset_index())
volatility["coefficient_of_variation"] = volatility.std_weekly_orders / volatility.mean_weekly_orders
volatility["max_to_mean"] = volatility.max_weekly_orders / volatility.mean_weekly_orders
most_variable = volatility.sort_values("coefficient_of_variation", ascending=False).head(10)
display(most_variable.round(3))

largest_volume = volatility.sort_values("mean_weekly_orders", ascending=False).head(10)
print("أعلى 10 وجبات في متوسط الطلب الأسبوعي:")
display(largest_volume.round(3))

,meal_id,weeks_observed,mean_weekly_orders,std_weekly_orders,max_weekly_orders,coefficient_of_variation,max_to_mean
34,2444,145,6500.469,7833.130,45010,1.205,6.924
25,1971,145,39507.724,43605.509,205806,1.104,5.209
10,1445,145,7304.972,7986.194,46095,1.093,6.310
49,2867,145,4923.621,4953.653,28314,1.006,5.751
35,2490,125,29673.000,27959.324,163333,0.942,5.504
30,2290,145,67381.400,60641.916,488252,0.900,7.246
16,1754,145,47021.097,41698.021,239884,0.887,5.102
27,2104,87,1838.586,1513.660,7453,0.823,4.054
15,1727,145,32661.979,26274.639,153238,0.804,4.692
13,1558,145,12061.655,9383.111,45057,0.778,3.736


أعلى 10 وجبات في متوسط الطلب الأسبوعي:


,meal_id,weeks_observed,mean_weekly_orders,std_weekly_orders,max_weekly_orders,coefficient_of_variation,max_to_mean
30,2290,145,67381.400,60641.916,488252,0.900,7.246
22,1885,145,57560.317,10998.042,105559,0.191,1.834
16,1754,145,47021.097,41698.021,239884,0.887,5.102
26,1993,145,45909.352,10707.706,96612,0.233,2.104
1,1109,145,43915.676,24620.200,169161,0.561,3.852
46,2707,145,40915.807,9760.739,67470,0.239,1.649
25,1971,145,39507.724,43605.509,205806,1.104,5.209
48,2826,145,35104.159,17059.038,123057,0.486,3.505
15,1727,145,32661.979,26274.639,153238,0.804,4.692
0,1062,145,32502.034,6382.437,50624,0.196,1.558


## الحدود

- يغطي التدريب الأسابيع المتاحة في الملف فقط؛ ولا توجد قيمة `num_orders` لأسابيع الاختبار.
- سجلات الطلب رصدية، وقد تتأثر بعوامل غير مقاسة مثل التوافر والتسعير ومزيج العملاء؛ لذلك لا تثبت النماذج السببية.
- مؤشرا البريد والصفحة ليسا عشوائيين بالضرورة. معاملات النموذج وفواصل الثقة تصف ارتباطات بعد ضبط التأثيرات المحددة في الكود.
- فرق السعر ليس خصمًا موثقًا، والعملة غير محددة؛ لا تفسر وحدات السعر نقديًا.
- لا يساوي تذبذب الطلب وحده كمية المخزون الاحتياطي المطلوبة. يلزم لذلك زمن التوريد، وسياسة مستوى الخدمة، وقيود التلف والطاقة، وبيانات نفاد المخزون.
- الأسابيع التي لا يظهر فيها سجل للوجبة مستبعدة من حساب التقلب ولا تعامل كطلب صفري.

## التنبؤ بالأسابيع 146–155 والتحقق الزمني

ينفذ القسم التالي انحدارًا خطيًا ويختبره تنبؤًا متحركًا أسبوعًا واحدًا في كل مرة على الأسابيع 136–145، مع مقارنة MAE بخط أساس الأسبوع السابق على الصفوف التي لها مشاهدة فعلية للزوج نفسه في الأسبوع السابق. بعد ذلك يعيد التدريب حتى الأسبوع 145 ويستخدم خصائص `test.csv` لإنتاج تنبؤات الأسابيع 146–155. تفاصيل الحساب والتحقق وسجلات التنبؤ والرسوم في `scripts/forecast_demand.py` ومخرجات `reports/client_report/`.

In [ ]:
from pathlib import Path
import runpy

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "scripts" / "forecast_demand.py").exists()
)
runpy.run_path(str(project_root / "scripts" / "forecast_demand.py"), run_name="__main__")